# Tool calling

[Back to the introduction](ch01_demo.ipynb). Run in order in a fresh kernel. A model chooses between registered Python functions, and `ToolRunner` checks every choice before anything runs. This loads Qwen2.5-0.5B-Instruct (q4, WASM, about 790 MB): it is a small model, so it can pick the wrong tool or produce invalid JSON. A test of a similar prompt found the right tool in 5 of 8 questions, so treat the model run below as a demonstration and not as a reliable assistant. The checking, the step limit and the record of what happened do not depend on the model, and the second half shows them with a scripted model, so every run of it gives the same result.

The tools here only calculate and read the clock. Nothing leaves the notebook, and the camera, microphone and network are not involved.

In [ ]:
from datetime import datetime

from lmlite import LLM, Tool, ToolRunner

A `Tool` is a name, a description the model reads, a function and a JSON Schema for its arguments. The schema is what the runner validates: the type, `enum`, `properties`, `required` and `items` are checked, and an argument the schema does not list is refused.

In [ ]:
def multiply(a, b):
    return a * b


def current_time():
    return datetime.now().strftime("%H:%M")


tools = [
    Tool(
        "multiply",
        "Multiply two numbers exactly.",
        multiply,
        {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]},
    ),
    Tool("current_time", "The current time of day, as HH:MM.", current_time),
]

In [ ]:
llm = await LLM.create(
    generator_model="onnx-community/Qwen2.5-0.5B-Instruct",
    backend="wasm",
    dtype="q4",
    max_new_tokens=80,
    return_full_text=False,
    do_sample=False,
)

`run()` asks the model, validates its reply, runs the tool and gives the result back, until the model answers or a limit is reached: `max_steps` replies, `timeout` seconds for the run, `tool_timeout` for one tool. Each reply costs seconds in WASM, so the limits are small.

In [ ]:
runner = ToolRunner(llm, tools, max_steps=4, timeout=180, tool_timeout=10)
result = await runner.run("What is 17 times 23?")
print(result.stopped, "->", result.answer)

`result.steps` records every reply and what became of it. `outcome` is `ok` for a tool that ran, `answer` for the final reply, `invalid` for a reply that was refused, `error` for a tool that failed, and `timeout` for the step a run timeout cut short.

In [ ]:
for step in result.steps:
    print(step["step"], step["outcome"], step.get("tool"), step.get("arguments"), "|", step["detail"])

## What happens to invalid output

A reply that names an unknown tool, has arguments the schema rejects or is not JSON never runs a tool. The reason goes back to the model as a message, which uses up one step, so a model that keeps failing ends at the step limit. A scripted model makes this repeatable: it returns four fixed replies, and the tool runs only for the last valid one.

In [ ]:
import json


class Scripted:
    """Stands in for a model: anything with an async chat(messages) works."""

    def __init__(self, *replies):
        self.replies = list(replies)
        self.seen = []

    async def chat(self, messages):
        self.seen.append(messages[-1]["content"])
        return self.replies.pop(0)


calls = []
counted = Tool("multiply", "Multiply two numbers.", lambda a, b: calls.append((a, b)) or a * b, tools[0].parameters)
scripted = Scripted(
    json.dumps({"tool": "divide", "arguments": {"a": 6, "b": 3}}),
    json.dumps({"tool": "multiply", "arguments": {"a": "six", "b": 3}}),
    "I think the answer is 18.",
    json.dumps({"tool": "multiply", "arguments": {"a": 6, "b": 3}}),
    json.dumps({"answer": "18"}),
)
checked = await ToolRunner(scripted, [counted], max_steps=5).run("What is 6 times 3?")
for step in checked.steps:
    print(step["step"], step["outcome"], "|", step["detail"])
print("The tool ran", len(calls), "time(s):", calls)

The step limit ends a model that never gets it right. The result has no answer and says why it stopped.

In [ ]:
stuck = await ToolRunner(Scripted("no", "still no", "no again"), [counted], max_steps=2).run("What is 6 times 3?")
print(stuck.stopped, stuck.answer, [step["outcome"] for step in stuck.steps])

In [ ]:
await llm.dispose()

The model is disposed without clearing cached downloads. The observation of a tool goes back to the model labelled as data, which limits the damage from a page that contains instructions but does not remove it: a small model can still be misled, so do not give a tool more reach than you would give the model's text. A tool that needs a photo or a recording can `await camera.request(...)`, which asks the user to tap first.